# 04 - Feature Engineering

### Notebook Overview

Create new features based on domain knowledge and EDA findings. Feature engineering is about giving the model better inputs to work with, not just more inputs.

**Input:** `../data/car-price-cleaned.csv`  
**Output:** `../data/car-price-featured.csv`

### 1 - Imports

In [1]:
# Core Libraries
import pandas as pd
import numpy as np

### 2 - Load Data

In [2]:
df = pd.read_csv("../data/car-price-cleaned.csv")
print(f"Starting shape: {df.shape}")
df.head()

Starting shape: (205, 24)


,symboling,fueltype,aspiration,doornumber,carbody,drivewheel,wheelbase,carlength,carwidth,carheight,...,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price,brand
0,3,gas,std,2,convertible,rwd,88.6,168.8,64.1,48.8,...,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0,alfa-romeo
1,3,gas,std,2,convertible,rwd,88.6,168.8,64.1,48.8,...,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0,alfa-romeo
2,1,gas,std,2,hatchback,rwd,94.5,171.2,65.5,52.4,...,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0,alfa-romeo
3,2,gas,std,4,sedan,fwd,99.8,176.6,66.2,54.3,...,mpfi,3.19,3.40,10.0,102,5500,24,30,13950.0,audi
4,2,gas,std,4,sedan,4wd,99.4,176.6,66.4,54.3,...,mpfi,3.19,3.40,8.0,115,5500,18,22,17450.0,audi


### 3 - Combine City and Highway MPG

EDA showed `citympg` and `highwaympg` are 0.97 correlated. Keeping both adds redundancy without new information. We combine them into a single `avg_mpg` feature, which is also how fuel economy is commonly reported (the EPA combined rating is a weighted average of city and highway).

In [3]:
df["avg_mpg"] = (df["citympg"] + df["highwaympg"]) / 2
df = df.drop(columns=["citympg", "highwaympg"])

print(f"avg_mpg range: {df['avg_mpg'].min():.1f} - {df['avg_mpg'].max():.1f}")
print(f"Correlation with price: {df['avg_mpg'].corr(df['price']):.3f}")

avg_mpg range: 15.0 - 51.5
Correlation with price: -0.697


### 4 - Power-to-Weight Ratio

In the car world, the horsepower-to-weight ratio is a standard measure of performance. A heavy car with a small engine feels sluggish; a light car with a big engine feels fast. This ratio captures something neither feature does alone, and performance is a key driver of price.

In [4]:
df["hp_per_weight"] = df["horsepower"] / df["curbweight"]

print(f"hp_per_weight range: {df['hp_per_weight'].min():.4f} - {df['hp_per_weight'].max():.4f}")
print(f"Correlation with price: {df['hp_per_weight'].corr(df['price']):.3f}")

hp_per_weight range: 0.0199 - 0.0856
Correlation with price: 0.534


### 5 - Brand Price Tier

EDA showed clear price separation by brand, but with 22 unique brands across 205 rows, encoding brand directly creates many sparse columns. Instead, we group brands into price tiers based on their median price in the dataset. This preserves the pricing signal with far fewer categories.

We use the data itself to define the tiers (rather than external knowledge about luxury brands) so the feature is grounded in what this dataset actually shows.

In [5]:
# Calculate median price per brand
brand_median = df.groupby("brand")["price"].median().sort_values(ascending=False)
print("Median price by brand:")
print(brand_median.to_string())

Median price by brand:
brand
jaguar        35550.0
buick         32892.0
porsche       32528.0
bmw           22835.0
volvo         18420.0
audi          17710.0
peugeot       16630.0
mercury       16503.0
alfa-romeo    16500.0
saab          15275.0
mazda         10595.0
volkswagen     9737.5
renault        9595.0
toyota         9103.0
isuzu          8916.5
mitsubishi     8499.0
nissan         8124.0
subaru         7894.0
dodge          7609.0
plymouth       7609.0
honda          7295.0
chevrolet      6295.0


In [6]:
# Define tiers using terciles of the brand median prices
tier_thresholds = brand_median.quantile([1/3, 2/3])
low_threshold = tier_thresholds.iloc[0]
high_threshold = tier_thresholds.iloc[1]

def assign_tier(brand):
    median_price = brand_median[brand]
    if median_price >= high_threshold:
        return "premium"
    elif median_price >= low_threshold:
        return "mid"
    else:
        return "budget"

df["brand_tier"] = df["brand"].map(assign_tier)

print(f"\nTier thresholds: budget < ${low_threshold:,.0f} <= mid < ${high_threshold:,.0f} <= premium")
print(f"\nTier distribution:")
print(df["brand_tier"].value_counts())
print(f"\nBrands per tier:")
for tier in ["premium", "mid", "budget"]:
    brands = sorted(df.loc[df["brand_tier"] == tier, "brand"].unique())
    print(f"  {tier}: {', '.join(brands)}")


Tier thresholds: budget < $8,916 <= mid < $16,503 <= premium

Tier distribution:
brand_tier
mid        76
budget     75
premium    54
Name: count, dtype: int64

Brands per tier:
  premium: audi, bmw, buick, jaguar, mercury, peugeot, porsche, volvo
  mid: alfa-romeo, isuzu, mazda, renault, saab, toyota, volkswagen
  budget: chevrolet, dodge, honda, mitsubishi, nissan, plymouth, subaru


In [7]:
# Drop brand now that we have brand_tier
df = df.drop(columns=["brand"])

### 6 - Verify Final State

In [8]:
print(f"Final shape: {df.shape}")
print(f"\nColumns:")
print(df.dtypes)
print(f"\nNull values: {df.isnull().sum().sum()}")

Final shape: (205, 24)

Columns:
symboling             int64
fueltype                str
aspiration              str
doornumber            int64
carbody                 str
drivewheel              str
wheelbase           float64
carlength           float64
carwidth            float64
carheight           float64
curbweight            int64
enginetype              str
cylindernumber        int64
enginesize            int64
fuelsystem              str
boreratio           float64
stroke              float64
compressionratio    float64
horsepower            int64
peakrpm               int64
price               float64
avg_mpg             float64
hp_per_weight       float64
brand_tier              str
dtype: object

Null values: 0


In [9]:
df.head()

,symboling,fueltype,aspiration,doornumber,carbody,drivewheel,wheelbase,carlength,carwidth,carheight,...,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,price,avg_mpg,hp_per_weight,brand_tier
0,3,gas,std,2,convertible,rwd,88.6,168.8,64.1,48.8,...,mpfi,3.47,2.68,9.0,111,5000,13495.0,24.0,0.043564,mid
1,3,gas,std,2,convertible,rwd,88.6,168.8,64.1,48.8,...,mpfi,3.47,2.68,9.0,111,5000,16500.0,24.0,0.043564,mid
2,1,gas,std,2,hatchback,rwd,94.5,171.2,65.5,52.4,...,mpfi,2.68,3.47,9.0,154,5000,16500.0,22.5,0.054552,mid
3,2,gas,std,4,sedan,fwd,99.8,176.6,66.2,54.3,...,mpfi,3.19,3.40,10.0,102,5500,13950.0,27.0,0.043646,premium
4,2,gas,std,4,sedan,4wd,99.4,176.6,66.4,54.3,...,mpfi,3.19,3.40,8.0,115,5500,17450.0,20.0,0.040722,premium


### 7 - Save Featured Data

In [10]:
df.to_csv("../data/car-price-featured.csv", index=False)
print("Saved to ../data/car-price-featured.csv")

Saved to ../data/car-price-featured.csv


### 8 - Summary

**New features created:**
- `avg_mpg`: average of `citympg` and `highwaympg` (r = -0.70 with price). Replaces two near-duplicate columns (r = 0.97 between them).
- `hp_per_weight`: horsepower / curbweight ratio (r = 0.53 with price). Captures car performance in a single metric.
- `brand_tier`: brands grouped into premium, mid, and budget tiers based on median price terciles. Reduces 22 brand categories to 3 tiers with balanced distribution (54 / 76 / 75).

**Columns removed:**
- `citympg`, `highwaympg` (replaced by `avg_mpg`)
- `brand` (replaced by `brand_tier`)

**Net result:** 24 → 24 columns (removed 3, added 3). 205 rows unchanged, zero nulls.